# 00 — Colab smoke test

**Goal:** confirm the free-Colab GPU environment works for gaussian-splatting-style code, end to end, in ~1 minute, with **no extra installs and no network**.

Fits a synthetic 2D image with rotated 2D gaussians using pure PyTorch — same optimisation pattern as 3D Shape-of-Motion, just stripped down.

**Before running:** `Runtime → Change runtime type → T4 GPU` (or any GPU).

Auto-shrinks the target image so it fits any GPU. Bump `TARGET_LONG_SIDE` later if you have headroom.

In [ ]:
!nvidia-smi -L

In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "Switch runtime to GPU."
print(torch.cuda.get_device_name(0), "|", torch.__version__)

## 1. Build a target image

Synthetic colourful pattern (six gaussian blobs on a vertical gradient) — fully deterministic, no network. `TARGET_LONG_SIDE=96` is the smoke-test size; fits any GPU.

In [ ]:
TARGET_LONG_SIDE = 96  # bump to 192 if you have an L4/A100

def make_target(side: int, seed: int = 0) -> np.ndarray:
    """Six soft colour blobs on a vertical gradient. Deterministic."""
    rng = np.random.default_rng(seed)
    yy, xx = np.meshgrid(
        np.linspace(-1, 1, side), np.linspace(-1, 1, side), indexing="ij"
    )
    img = np.zeros((side, side, 3), dtype=np.float32)
    for _ in range(6):
        cx, cy = rng.uniform(-0.7, 0.7, size=2)
        rad = rng.uniform(0.15, 0.30)
        color = rng.uniform(0.2, 1.0, size=3).astype(np.float32)
        d2 = (xx - cx) ** 2 + (yy - cy) ** 2
        img += np.exp(-d2 / (2 * rad ** 2))[..., None] * color
    bg = 0.2 + 0.3 * (yy * 0.5 + 0.5)[..., None] * np.array(
        [0.3, 0.5, 0.8], dtype=np.float32
    )
    return np.clip(img + bg, 0.0, 1.0)

target = torch.tensor(make_target(TARGET_LONG_SIDE), dtype=torch.float32, device=device)
H, W, _ = target.shape
print(f"target: {H}x{W}")
plt.imshow(target.cpu()); plt.axis("off"); plt.title("target"); plt.show()

## 2. 2D gaussian parameters + a tiny rasterizer

Each gaussian has: position `xy`, log-scale `log_scale`, rotation `rot`, color `rgb`, opacity `alpha`. Rendering is a soft-alpha mean over all gaussians at each pixel. Naive but differentiable and short.

In [ ]:
N = 1500  # number of gaussians
torch.manual_seed(0)

xy = nn.Parameter(torch.rand(N, 2, device=device) * torch.tensor([W, H], device=device))
log_scale = nn.Parameter(torch.full((N, 2), math.log(2.0), device=device))
rot = nn.Parameter(torch.zeros(N, device=device))
rgb_logit = nn.Parameter(torch.randn(N, 3, device=device) * 0.1)
opacity_logit = nn.Parameter(torch.zeros(N, device=device))

yy, xx = torch.meshgrid(
    torch.arange(H, device=device, dtype=torch.float32),
    torch.arange(W, device=device, dtype=torch.float32),
    indexing="ij",
)

def render():
    dx = xx[None] - xy[:, 0, None, None]
    dy = yy[None] - xy[:, 1, None, None]
    c, s = torch.cos(rot)[:, None, None], torch.sin(rot)[:, None, None]
    dxr = c * dx + s * dy
    dyr = -s * dx + c * dy
    sx, sy = log_scale.exp()[:, 0, None, None], log_scale.exp()[:, 1, None, None]
    g = torch.exp(-0.5 * ((dxr / sx) ** 2 + (dyr / sy) ** 2))
    a = torch.sigmoid(opacity_logit)[:, None, None] * g  # (N, H, W)
    rgb = torch.sigmoid(rgb_logit)  # (N, 3)
    w_sum = a.sum(dim=0).clamp_min(1e-6)
    return (a[..., None] * rgb[:, None, None, :]).sum(dim=0) / w_sum[..., None]

## 3. Fit

In [ ]:
ITERS = 400
opt = torch.optim.Adam(
    [
        {"params": [xy], "lr": 0.5},
        {"params": [log_scale, rot], "lr": 0.05},
        {"params": [rgb_logit, opacity_logit], "lr": 0.05},
    ]
)

losses = []
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
for it in range(ITERS):
    opt.zero_grad()
    pred = render()
    loss = F.mse_loss(pred, target)
    loss.backward()
    opt.step()
    losses.append(loss.item())
    if it % 50 == 0 or it == ITERS - 1:
        print(f"iter {it:4d}  loss {loss.item():.4f}")
dt = time.time() - t0
print(f"\ndone in {dt:.1f}s  ({ITERS/dt:.0f} it/s)")
print(f"peak VRAM: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

In [ ]:
with torch.no_grad():
    final = render().cpu()

fig, ax = plt.subplots(1, 3, figsize=(11, 3))
ax[0].imshow(target.cpu()); ax[0].set_title("target"); ax[0].axis("off")
ax[1].imshow(final.clamp(0, 1)); ax[1].set_title(f"fit (N={N})"); ax[1].axis("off")
ax[2].plot(losses); ax[2].set_title("loss"); ax[2].set_yscale("log")
plt.tight_layout(); plt.show()

psnr = -10 * math.log10(((final.to(device) - target) ** 2).mean().item())
print(f"PSNR: {psnr:.2f} dB")

## 4. If this worked → you're ready for Phase 1

What this proved:
- Colab GPU is alive and PyTorch sees it.
- A gaussian-splatting-style **differentiable rasteriser → optimiser → loss** loop runs end to end.
- Memory / wall-clock are in the right ballpark for the toy size.

What this did **not** prove:
- That `gsplat` (the proper CUDA-accelerated 3D rasteriser) compiles on Colab — that needs a real install, expect ~5–10 min the first time.
- That Shape-of-Motion's training pipeline works — that's Phase 1, will live in a separate notebook (`01_phase1_som_smoke.ipynb`).
- Anything about **video diffusion / SDS** — Phase 3, won't fit on free T4.